# 🐖 Notebook 13 | Predicciones de validación para calibración probabilística

**RECURSO: GPU T4. Solo inferencia; no entrena ninguna red neuronal.**

**Objetivo:** congelar predicciones y máscaras de referencia en las **196 imágenes de validación (4 fechas)** con el mismo protocolo del Notebook 09. El Notebook 14 (CPU) ajustará una calibración probabilística **solo en validación** y la evaluará reutilizando los archivos del **test de 373 imágenes ya evaluado en Notebook 09**.

> **Advertencia científica:** calibramos la **confiabilidad de las detecciones individuales** (probabilidad de ser una detección verdadera con IoU de máscara ≥0,50), **no** incertidumbre de píxel ni probabilidad de que un cerdo exista. Los falsos negativos sin detección no tienen puntuación de confianza y no entran en el ECE; se documentan por separado. Este test ya se ha consultado previamente: es una evaluación post hoc exploratoria, no un test nuevo y ciego.

Si se interrumpe, la inferencia por imagen se guarda en archivos `.jsonl` de avance: **no borres estos archivos**.

## A. Preparar GPU y dependencias

In [ ]:
import torch
assert torch.cuda.is_available(), 'Activa T4 GPU antes de ejecutar (Entorno de ejecución > Cambiar tipo).'
print('✅ GPU:',torch.cuda.get_device_name(0))


In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "pandas>=2,<3" openpyxl pillow opencv-python-headless


## B. Datos congelados, pesos y trazabilidad

In [ ]:
from google.colab import drive, files as colab_files
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,io,os,zipfile,cv2,yaml,numpy as np,pandas as pd
from PIL import Image
import torch,torchvision
from ultralytics import YOLO
from pycocotools import mask as maskutils
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(),f'Falta carpeta del proyecto: {ROOT}'
RES=ROOT/'03_Resultados';DATA=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
REPORTS=RES/'05_Reportes';METRICS=RES/'03_Metricas';EXPORT=ROOT/'05_Exportaciones_ZIP'
SAVE=REPORTS/'CERDOS_13_Calibracion_Validacion'
for d in [SAVE,METRICS,EXPORT]:d.mkdir(parents=True,exist_ok=True)
YAML=DATA/'data_fecha.yaml';MANIFEST=DATA/'particion_fecha_manifest.json'
YW=RES/'06_Entrenamiento'/'YOLO11n_fecha_v1'/'weights'/'best.pt'
MW=RES/'06_Entrenamiento'/'MaskRCNN_fecha_v1'/'maskrcnn_best_model.pt'
P09=REPORTS/'CERDOS_09_COCO_Armonizado'/'CERDOS_09_Protocolo.json'
for f in [YAML,MANIFEST,YW,MW,P09]:
    if not f.is_file():raise FileNotFoundError(f'Dependencia ausente: {f}')
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

proto09=json.loads(P09.read_text(encoding='utf-8'))
assert proto09['yolo_sha256']==sha256(YW),'Los pesos YOLO cambiaron frente al 09'
assert proto09['mask_sha256']==sha256(MW),'Los pesos Mask R-CNN cambiaron frente al 09'
assert proto09['yaml_sha256']==sha256(YAML),'El YAML cambió frente al 09'
conf=yaml.safe_load(YAML.read_text(encoding='utf-8'))
paths=[Path(x.strip()) for x in Path(conf['val']).read_text(encoding='utf-8').splitlines() if x.strip()]
assert len(paths)==196 and len(set(map(str,paths)))==196 and all(x.is_file() for x in paths),'Validación incompleta o diferente de 196 imágenes'
assign=json.loads(MANIFEST.read_text(encoding='utf-8'))['asignaciones_fecha']
assert len({p.name[:10] for p in paths})==4
assert all(assign.get(p.name[:10])=='val' for p in paths),'Imágenes no pertenecen al split val congelado'
assert set(p.name[:10] for p in paths).isdisjoint({k for k,v in assign.items() if v=='test'}),'Fuga temporal entre val y test'
PROTO={'version':'CERDOS_13_CAL_VAL_v1','subset':'val','n_images':196,'n_dates':4,
       'val_list_sha256':sha256(Path(conf['val'])),'yaml_sha256':sha256(YAML),
       'yolo_sha256':sha256(YW),'mask_sha256':sha256(MW),
       'yolo_imgsz':proto09['yolo_imgsz'],'yolo_conf':proto09['yolo_conf'],
       'yolo_iou_nms':proto09['yolo_iou_nms'],'retina_masks':proto09['yolo_retina_masks'],
       'max_dets':proto09['max_dets'],'mask_score_thresh':proto09['mask_score_thresh'],
       'mask_bin_thresh':proto09['mask_bin_thresh'],'raster':proto09['gt_raster']}
PROTO_FILE=SAVE/'CERDOS_13_Protocolo.json'
if PROTO_FILE.exists():
    if json.loads(PROTO_FILE.read_text(encoding='utf-8'))!=PROTO:raise RuntimeError('Protocolo 13 existente incompatible; se detiene para evitar mezclar experimentos.')
else:
    if any(SAVE.iterdir()):raise RuntimeError('Carpeta 13 ya tiene archivos sin protocolo; revisar antes de reintentar.')
    PROTO_FILE.write_text(json.dumps(PROTO,indent=2,ensure_ascii=False),encoding='utf-8')
torch.manual_seed(2026);np.random.seed(2026)
print('✅ VALIDACIÓN:',len(paths),'imágenes | GPU:',torch.cuda.get_device_name(0))
print('✅ Pesos y particiones verificados; test no se consulta ni modifica en este notebook.')


## C. Anotaciones COCO de validación (mismo procedimiento que 09)

In [ ]:
GT_FILE=SAVE/'CERDOS_13_GT_val_coco.json'
def encoded(mask):
    r=maskutils.encode(np.asfortranarray(mask.astype(np.uint8)))
    r['counts']=r['counts'].decode('ascii')
    return r

def make_gt():
    images=[];anns=[];idx=0
    for iid,p in enumerate(paths,1):
        with Image.open(p) as im:w,h=im.size
        images.append({'id':iid,'width':w,'height':h,'file_name':p.name})
        lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        if not lab.is_file():raise FileNotFoundError(f'Anotación ausente: {lab}')
        for row in lab.read_text(encoding='utf-8-sig').splitlines():
            if not row.strip():continue
            words=row.split()
            assert int(words[0])==0,'Clase inesperada'
            xy=np.asarray([float(z) for z in words[1:]],np.float32)
            assert len(xy)>=6 and len(xy)%2==0 and np.isfinite(xy).all() and np.all((xy>=0)&(xy<=1))
            coords=np.rint(xy.reshape(-1,2)*[w,h]).astype(np.int32)
            coords[:,0]=np.clip(coords[:,0],0,w-1);coords[:,1]=np.clip(coords[:,1],0,h-1)
            m=np.zeros((h,w),np.uint8);cv2.fillPoly(m,[coords],1)
            assert m.sum()>=3
            idx+=1;rle=encoded(m)
            anns.append({'id':idx,'image_id':iid,'category_id':1,'segmentation':rle,
                         'bbox':[float(z) for z in maskutils.toBbox(rle)],'area':int(m.sum()),'iscrowd':0})
    return {'info':{'description':'FaroPigSeg validación por fecha: calibración, no test'},'licenses':[],
            'images':images,'annotations':anns,'categories':[{'id':1,'name':'cerdo','supercategory':'animal'}]}

if GT_FILE.exists():gt=json.loads(GT_FILE.read_text(encoding='utf-8'))
else:
    gt=make_gt()
    GT_FILE.write_text(json.dumps(gt,ensure_ascii=False),encoding='utf-8')
assert len(gt['images'])==196 and len(gt['annotations'])==1995,'Anotaciones de validación inesperadas; deténgase.'
assert [x['file_name'] for x in gt['images']]==[p.name for p in paths],'Orden de imágenes diferente'
print('✅ GT validación:',len(gt['images']),'imágenes;',len(gt['annotations']),'máscaras')


## D. YOLO11n-seg: inferencia en validación con recuperación por imagen

In [ ]:
YFILE=SAVE/'CERDOS_13_Pred_YOLO_val_coco.json'
YCP=SAVE/'avance_YOLO_val.jsonl'
MFILE=SAVE/'CERDOS_13_Pred_MaskRCNN_val_coco.json'
MCP=SAVE/'avance_MaskRCNN_val.jsonl'

def read_checkpoint(path):
    result={}
    if not path.exists():return result
    with open(path,'r',encoding='utf-8') as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            try:item=json.loads(line)
            except json.JSONDecodeError:raise RuntimeError(f'Checkpoint con línea incompleta: {path}, línea {lineno}. No borrar; recuperar antes de continuar.')
            i=int(item['image_id'])
            assert 1<=i<=len(paths) and i not in result,f'Checkpoint duplicado/inválido {i}'
            result[i]=item['rows']
    return result

def infer_resume(final_path,cp_path,model_name,predict_one):
    if final_path.is_file():
        obj=json.loads(final_path.read_text(encoding='utf-8'))
        print('✅ Predicciones anteriores completas:',model_name,len(obj));return obj
    done=read_checkpoint(cp_path)
    print(model_name,'| imágenes ya guardadas:',len(done),'/',len(paths))
    for iid,p in enumerate(paths,1):
        if iid in done:continue
        result=predict_one(iid,p)
        assert all(int(x['image_id'])==iid for x in result)
        with open(cp_path,'a',encoding='utf-8') as f:
            f.write(json.dumps({'image_id':iid,'rows':result},ensure_ascii=False)+'\n')
        done[iid]=result
        if iid%25==0:print(model_name,'completado:',iid,'/',len(paths))
    assert len(done)==len(paths),'Inferencia incompleta'
    obj=[row for i in range(1,len(paths)+1) for row in done[i]]
    final_path.write_text(json.dumps(obj,ensure_ascii=False),encoding='utf-8')
    return obj

if not YFILE.exists():
    detector=YOLO(str(YW))
    def yolo_one(i,p):
        prediction=detector.predict(source=str(p),device=0,imgsz=PROTO['yolo_imgsz'],
            conf=PROTO['yolo_conf'],iou=PROTO['yolo_iou_nms'],max_det=PROTO['max_dets'],
            retina_masks=True,verbose=False,save=False)[0]
        scores=prediction.boxes.conf.detach().cpu().numpy() if prediction.boxes is not None else np.empty(0)
        with Image.open(p) as im:w,h=im.size
        if len(scores)==0:return []
        assert prediction.masks is not None and prediction.masks.data is not None
        masks=prediction.masks.data.detach().cpu().numpy()
        assert masks.shape==(len(scores),h,w),f'Máscaras YOLO incompatibles: {masks.shape}'
        return [{'image_id':i,'category_id':1,'segmentation':encoded(m>0.5),'score':float(s)}
                for s,m in zip(scores,masks) if (m>0.5).any()]
    ypred=infer_resume(YFILE,YCP,'YOLO',yolo_one)
    del detector;torch.cuda.empty_cache()
else:ypred=infer_resume(YFILE,YCP,'YOLO',None)
print('✅ Detecciones YOLO en validación:',len(ypred))


## E. Mask R-CNN: inferencia en validación con recuperación por imagen

In [ ]:
def build_maskrcnn():
    m=maskrcnn_resnet50_fpn(weights=None,weights_backbone=None,
        min_size=512,max_size=768,trainable_backbone_layers=3)
    n=m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor=FastRCNNPredictor(n,2)
    z=m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor=MaskRCNNPredictor(z,256,2)
    m.roi_heads.score_thresh=PROTO['mask_score_thresh']
    m.roi_heads.detections_per_img=PROTO['max_dets']
    return m

if not MFILE.exists():
    mask_model=build_maskrcnn().to('cuda:0')
    checkpoint=torch.load(MW,map_location='cpu',weights_only=False)
    mask_model.load_state_dict(checkpoint['model'],strict=True)
    del checkpoint;mask_model.eval()
    def mask_one(i,p):
        with Image.open(p) as im:rgb=np.asarray(im.convert('RGB')).copy()
        h,w=rgb.shape[:2]
        with torch.inference_mode():out=mask_model([to_tensor(rgb).to('cuda:0')])[0]
        scores=out['scores'].detach().cpu().numpy()
        if not len(scores):return []
        masks=out['masks'].detach().cpu().numpy()[:,0]
        assert masks.shape==(len(scores),h,w),f'Máscaras Mask R-CNN incompatibles: {masks.shape}'
        return [{'image_id':i,'category_id':1,'segmentation':encoded(m>PROTO['mask_bin_thresh']),
                 'score':float(s)} for s,m in zip(scores,masks) if (m>PROTO['mask_bin_thresh']).any()]
    mpred=infer_resume(MFILE,MCP,'Mask R-CNN',mask_one)
    del mask_model;torch.cuda.empty_cache()
else:mpred=infer_resume(MFILE,MCP,'Mask R-CNN',None)
print('✅ Detecciones Mask R-CNN en validación:',len(mpred))


## F. Verificación y ZIP del Notebook 13

In [ ]:
assert all(int(r['image_id']) in set(range(1,197)) for r in ypred+mpred)
assert all(0<=float(r['score'])<=1 and 'segmentation' in r for r in ypred+mpred)
SUMMARY={'etapa':'13_inferencia_de_validacion_sin_entrenamiento',
         'fecha_utc':datetime.now(timezone.utc).isoformat(),'protocolo':PROTO,
         'imagenes_val':196,'instancias_gt_val':len(gt['annotations']),
         'predicciones_val_YOLO':len(ypred),'predicciones_val_MaskRCNN':len(mpred),
         'nota':'Calibración en Notebook 14; no se ajustan modelos en este notebook.'}
# Excel simple para inspeccionar sin abrir archivos COCO/RLE.
image_date={i:p.name[:10] for i,p in enumerate(paths,1)}
rows=[]
for model,items in [('YOLO11n-seg',ypred),('Mask R-CNN',mpred)]:
    counts={d:0 for d in sorted(set(image_date.values()))}
    for r in items:counts[image_date[int(r['image_id'])]]+=1
    for date,count in counts.items():
        rows.append({'modelo':model,'fecha_validacion':date,
                     'numero_imagenes':sum(v==date for v in image_date.values()),
                     'instancias_predichas':count})
XLSX=SAVE/'CERDOS_13_Resumen_Validacion.xlsx'
with pd.ExcelWriter(XLSX,engine='openpyxl') as writer:
    pd.DataFrame(rows).to_excel(writer,sheet_name='Predicciones_Fecha',index=False)
    pd.DataFrame([PROTO]).to_excel(writer,sheet_name='Protocolo',index=False)
SUMFILE=SAVE/'CERDOS_13_Resumen.json'
SUMFILE.write_text(json.dumps(SUMMARY,ensure_ascii=False,indent=2),encoding='utf-8')
files_to_export=[PROTO_FILE,GT_FILE,YFILE,MFILE,SUMFILE,XLSX]
assert all(p.is_file() for p in files_to_export)
ZIP=EXPORT/'CERDOS_13_Predicciones_Validacion.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in files_to_export:z.write(p,arcname=p.name)
with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None
print('✅ Guardados archivos en Drive; ZIP:',ZIP)
print('✅ No se realizó ningún entrenamiento ni se abrió el test para calibrar.')
colab_files.download(str(ZIP))
